In [1]:
#!/usr/bin/python
import os,sys
folder_path = os.getcwd()
if folder_path not in sys.path:
	sys.path.append(folder_path)

from Compiler.Klayout_compiler import *
from Compiler.HFSS_compiler import *
from Compiler.Q3D_complier import *
from Builds.Build_universal_functions import *

In [8]:
def Set_substrate(object, metal_thickness=0.1, sub_size_x=1000, sub_size_y = 1800, sub_thickness = 525, z_spacing_above = 3000, z_spacing_below = 2600,):
    '''Unit (um)'''
    object.metal_thickness = metal_thickness
    object.sub_size_x = sub_size_x
    object.sub_size_y = sub_size_y
    object.sub_thickness = -sub_thickness
    # Draw Substrate: Si
    object.sub_name = 'silicon'
    object.substrate(dx=object.sub_size_x / 2, dy=object.sub_size_y / 2)
    # Draw Vacuum: vaccum
    z_above = z_spacing_above # Measured 05012026, Valla Fatemi Lab package, space above chip top surface.
    z_below = z_spacing_below + abs(object.sub_thickness) # Measured 05012026, Valla Fatemi Lab package, space below chip top surface.
    object.vacuum = object.modeler.create_box(
            origin=[-object.sub_size_x / 2, -object.sub_size_y / 2, (z_above-z_below)/2], 
            sizes=[object.sub_size_x, -object.sub_size_y, z_above + z_below],
            name="vacuum",
            material="vacuum")
    # Draw Gnd
    object.gnd = [object.line(start_x=-object.sub_size_x / 2, start_y=0, end_x=object.sub_size_x / 2, end_y=0, width=object.sub_size_y, name='Gnd')]
def R20_Optimize_transmon(object,):
    pass
def Boulder_Resonators(object,short_x_list=[400],short_y_list=[-800],shape_list=['x76.5,y1409,x-300,y-1030.3,x-300,y1065.5'],radius=75):
    """
    short_x_list: x position of resonator shorted end
    short_y_list: y position of resonator shorted end
    shape_list: the position/turning of the resonator
    radius: the radius of the turning
    """
    object.reson_width = 6
    object.reson_gap = 3
    object.openendTogap_ratio = 5/3 # Stupid way to say the open end to gnd spacing being 5 um
    object.toBeRemove = []
    object.toBeAdd = []
    open_x_list=[]
    open_y_list=[]
    for idx, (x, y, shape) in enumerate(zip(short_x_list,short_y_list,shape_list)):
        x_list, y_list = construct(start_x=x, start_y=y, shape=shape)
        open_x_list += [x_list[-1]]
        open_y_list += [y_list[-1]]
        center_line, trench = object.CPW_line(x_list.copy(), y_list.copy(), 
                        width=object.reson_width, gap=object.reson_gap, 
                        radius=radius, name='reson',
                        end=[0,1]) # 0: short end, 1: open end
        object.modeler.subtract(trench, [center_line], keep_originals=True) # get trench = gap, center_line
        object.fillet_corners(trench, 2.5) 
        object.fillet_corners(center_line, 2.5)
        object.toBeRemove += [trench,center_line]
        center_line_1, trench_1 = object.CPW_line(x_list.copy(), y_list.copy(), 
                        width=object.reson_width, gap=object.reson_gap, 
                        radius=radius, name='reson',
                        end=[0,1])
        object.fillet_corners(center_line_1, 3)
        object.toBeAdd += [center_line_1]
    object.gnd = object.modeler.subtract(object.gnd, object.toBeRemove, keep_originals=False)
    object.toBeAdd += [object.gnd]
    object.gnd = object.modeler.unite(object.toBeAdd)
    return open_x_list, open_y_list


'''Q3D '''
'''Klayout '''
folder_path = r'C:\Users\xd255\Documents\xd\20251106_Transmon_Simulation\Ouput'
file_name = '20260504_test'
project_name = os.path.join(folder_path, file_name)
start = time.time()
h = KLAYOUT(project_name)
h.extention='.gds'
Set_substrate(h,sub_size_x=6500, sub_size_y = 7000)
spacing=300
short_x_list = np.array([-2500,0,2500,2500,0,-2500])
short_y_list = np.array([10,10,10,-10,-10,-10])
short_x_list = short_x_list + spacing/2 * np.sign(short_y_list)
shape_list = ['x76.5,y1409,x-300,y-1030.3,x-300,y1065.5',
              'x76.5,y1409,x-300,y-1030.3,x-300,y1065.5',
              'x76.5,y1409,x-300,y-1030.3,x-300,y1065.5',
              'x-76.5,y-1409,x300,y1030.3,x300,y-1065.5',
              'x-76.5,y-1409,x300,y1030.3,x300,y-1065.5',
              'x-76.5,y-1409,x300,y1030.3,x300,y-1065.5',
              ]
Boulder_Resonators(h,short_x_list,short_y_list,shape_list)
h.tempsave()
print('time spent:', time.time()-start, 's')

[Info] KLAYOUT: Successfully filleted Region with radius 2.5um (2500 dbu).
[Info] KLAYOUT: Successfully filleted Region with radius 2.5um (2500 dbu).
[Info] KLAYOUT: Successfully filleted Region with radius 3um (3000 dbu).
[Info] KLAYOUT: Successfully filleted Region with radius 2.5um (2500 dbu).
[Info] KLAYOUT: Successfully filleted Region with radius 2.5um (2500 dbu).
[Info] KLAYOUT: Successfully filleted Region with radius 3um (3000 dbu).
[Info] KLAYOUT: Successfully filleted Region with radius 2.5um (2500 dbu).
[Info] KLAYOUT: Successfully filleted Region with radius 2.5um (2500 dbu).
[Info] KLAYOUT: Successfully filleted Region with radius 3um (3000 dbu).
[Info] KLAYOUT: Successfully filleted Region with radius 2.5um (2500 dbu).
[Info] KLAYOUT: Successfully filleted Region with radius 2.5um (2500 dbu).
[Info] KLAYOUT: Successfully filleted Region with radius 3um (3000 dbu).
[Info] KLAYOUT: Successfully filleted Region with radius 2.5um (2500 dbu).
[Info] KLAYOUT: Successfully fill

In [ ]:
# from Builds.Build_Boulder import *
# folder_path = r'C:\Users\xd255\Documents\xd\20251106_Transmon_Simulation\Models'
# file_name = 'Boulder_Resonator_Q3D'
# project_name = os.path.join(folder_path, file_name)
# start = time.time()
# '''Q3D '''
# h = Q3D(file_name)
# Build_Boulder(h)
# print('time spent:', time.time()-start, 's')
# '''Klayout '''
# h = KLAYOUT(project_name)
# h.extention='.gds'
# Build_Boulder(h)
# print('time spent:', time.time()-start, 's')

PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.17.4.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\xd255\AppData\Local\Temp\8\pyaedt_xd255_490afae3-42a2-4e74-b501-81162df3e0f3.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: Found active AEDT gRPC session on port 50051.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v252\AnsysEM
PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.17.4.
PyAEDT INFO: Returning found Desktop session with PID 32544!
PyAEDT INFO: Project Boulder_Resonator_Q3D set to active.
PyAEDT INFO: Active Design set to 1
PyAEDT INFO: Activ

'Klayout '

In [ ]:
# from Builds.Build_100nH import *
# folder_path = r'D:\OneDrive - Washington University in St. Louis\wustl\HLab\Project_gPhotonDetector\Design'
# file_name = '100nH_generated'
# project_name = os.path.join(folder_path, file_name)
# start = time.time()
# h = KLAYOUT(project_name)
# # h = HFSS(project_name)
# h.trap = 0
# Build_SingleFilter(h,lead_number=1)
# print('time spent:', time.time()-start, 's')

In [ ]:
# from Builds.Build_SD013_test import *
# folder_path = r'C:\Users\xingrui\OneDrive - Washington University in St. Louis\wustl\HLab\Project_MLGM\Designs\SD013test'
# file_name = 'SD013test_bareline_2025'
# project_name = os.path.join(folder_path, file_name)
# start = time.time()
# h = HFSS(file_name)
# h.trap = 0
# Build_013_test(h,lead_number=0)
# print('time spent:', time.time()-start, 's')

In [ ]:
# from Builds.Build_SD009 import *
# folder_path = r'/Users/chellybone/Library/CloudStorage/OneDrive-WashingtonUniversityinSt.Louis/wustl/HLab/Project_MLGM/Fab_files/SD_010'
# file_name = 'Test_009'
# project_name = os.path.join(folder_path, file_name)
# start = time.time()
# # h = HFSS(project_name)
# # h.trap = False
# # Build_hBN_waveguide(h,)
# h = KLAYOUT(project_name)
# h.trap = True
# Build_009(h,8)
# print('time spent:', time.time()-start, 's')

In [ ]:
# from Builds.Build_SD011_debug import *
# from Builds.Build_SD014 import *
# # folder_path = r'C:\Users\duxin\OneDrive - Washington University in St. Louis\wustl\HLab\Project_MLGM\Fab_files\SD_011'
# folder_path = r'C:\Users\xingrui\OneDrive - Washington University in St. Louis\wustl\HLab\Project_MLGM\Designs\SD014'
# file_name = 'temp_sim'
# project_name = os.path.join(folder_path, file_name)
# start = time.time()
# # h = KLAYOUT(project_name)
# h = HFSS(project_name)
# h.trap = 0
# Build_Utype(h,lead_number=3)
# print('time spent:', time.time()-start, 's')

In [ ]:
# from Builds.Build_SD012 import *
# folder_path = r'C:\Users\xingrui\OneDrive - Washington University in St. Louis\wustl\HLab\Project_MLGM\Fab_files\SD_012'
# file_name = 'SD012_removePad'
# project_name = os.path.join(folder_path, file_name)
# start = time.time()
# # h = KLAYOUT(project_name)
# h = HFSS(project_name)
# h.trap = 0
# Build_012(h,lead_number=2)
# print('time spent:', time.time()-start, 's')

In [ ]:
# from Builds.Build_SD013 import *
# folder_path = r'C:\Users\xingrui\OneDrive - Washington University in St. Louis\wustl\HLab\Project_MLGM\Designs\SD013'
# file_name = 'temp_sim'

# # folder_path = r'C:\Users\xingrui\OneDrive - Washington University in St. Louis\wustl\HLab\Project_MLGM\Fab_files\SD_013'
# # file_name = 'SD013_121123'
# project_name = os.path.join(folder_path, file_name)
# start = time.time()

# h.trap = 0
# Build_013(h,lead_number=2)
# print('time spent:', time.time()-start, 's')

In [ ]:
# from Builds.Build_100nH import *
# # folder_path = r'D:\OneDrive - Washington University in St. Louis\wustl\HLab\Project_gPhotonDetector\Design'
# file_name = '100nH_generated'
# project_name = os.path.join(folder_path, file_name)
# start = time.time()
# # h = KLAYOUT(project_name)
# h = HFSS(file_name)
# # h.trap = 0
# Build_SingleFilter(h,lead_number=1)
# print('time spent:', time.time()-start, 's')